# Question 1 Evaluation Results and Error Analysis

## 1. Initial Evaluation on 200 Variants

I ran the Question 1 evaluation script on all 200 variants. The initial evaluation results are shown below.

Overall, the model demonstrated very strong performance on most criteria. The majority of criteria achieved perfect or near-perfect performance, suggesting that the model is generally capable of correctly identifying and applying ACMG/AMP evidence criteria.

### Initial Per-Criterion Performance

| Criterion | N | Accuracy | Precision | Recall | F1 |
|-----------|---|----------|-----------|--------|----|
| BA1 | 3 | 1.000 | 1.000 | 1.000 | 1.000 |
| BP4 | 4 | 1.000 | 1.000 | 1.000 | 1.000 |
| BS1 | 8 | 1.000 | 1.000 | 1.000 | 1.000 |
| PM1 | 3 | 1.000 | 1.000 | 1.000 | 1.000 |
| PM2_SUPPORTING | 34 | 1.000 | 1.000 | 1.000 | 1.000 |
| PM4 | 1 | 1.000 | 1.000 | 1.000 | 1.000 |
| PM5 | 4 | 1.000 | 1.000 | 1.000 | 1.000 |
| PP3 | 28 | 0.964 | 1.000 | 0.941 | 0.970 |
| PS1 | 1 | 0.000 | 0.000 | 0.000 | 0.000 |
| PS3 | 1 | 0.000 | 0.000 | 0.000 | 0.000 |
| PS4 | 14 | 1.000 | 1.000 | 1.000 | 1.000 |
| PVS1 | 3 | 1.000 | 1.000 | 1.000 | 1.000 |

The only apparent weakness was PS1, where the model achieved 0 performance. However, the sample size for PS1 was only one example, meaning that no meaningful conclusion can be drawn from this result.

## 2. Limitations of the Original Evaluation Script

After examining the evaluation methodology, I identified a potential issue with the original evaluation script.

The original script only evaluated variant-criterion pairs where the "not applied" status was explicitly specified in the gold CSV file. However, many datasets only specify the criteria that **apply**, assuming that all other criteria not mentioned are implicitly **not applied**.

As a result, the original evaluation script excluded many negative examples, causing the evaluation sample size for many criteria to become extremely small.

This creates a misleading evaluation because the model is mostly tested on positive examples rather than being evaluated on its ability to correctly identify both applied and not-applied criteria.

Therefore, I modified the evaluation script to include all variant-criterion pairs and treat unspecified criteria as not applied. After this modification, I reran the evaluation.

## 3. Updated Evaluation Results

The updated evaluation results are shown below.

| Criterion | N | Accuracy | Precision | Recall | F1 |
|-----------|---|----------|-----------|--------|----|
| BA1 | 199 | 1.000 | 1.000 | 1.000 | 1.000 |
| BP4 | 138 | 0.659 | 0.078 | 1.000 | 0.145 |
| BP7 | 33 | 0.030 | 0.000 | 0.000 | 0.000 |
| BS1 | 199 | 0.980 | 0.600 | 1.000 | 0.750 |
| BS3 | 200 | 1.000 | 0.000 | 0.000 | 0.000 |
| PM1 | 90 | 0.978 | 0.600 | 1.000 | 0.750 |
| PM2_SUPPORTING | 200 | 0.250 | 0.185 | 1.000 | 0.312 |
| PM4 | 3 | 0.333 | 0.333 | 1.000 | 0.500 |
| PM5 | 90 | 0.711 | 0.133 | 1.000 | 0.235 |
| PP3 | 138 | 0.833 | 0.421 | 0.941 | 0.582 |
| PS1 | 90 | 0.989 | 0.000 | 0.000 | 0.000 |
| PS3 | 200 | 0.995 | 0.000 | 0.000 | 0.000 |
| PS4 | 199 | 0.894 | 0.400 | 1.000 | 0.571 |
| PVS1 | 53 | 0.057 | 0.057 | 1.000 | 0.107 |

## 4. Analysis of Updated Results

The updated evaluation reveals a much clearer picture of model behavior.

The model performs very well on several criteria:

- BA1
- BS1
- BS3
- PM1
- PS1
- PS3

However, for several criteria, the prototype essentially misclassifies most examples. The criteria with poor performance include:

- BP7
- BP4
- PM2_SUPPORTING
- PM4
- PM5
- PP3
- PS4
- PVS1

In particular, BP7 shows extremely poor performance, with an F1 score of 0. This indicates that the model currently struggles significantly with correctly deciding when BP7 should or should not be applied.

## 5. Preliminary Error Analysis

After manually examining the errors, I identified three major categories of failure.

### 5.1 Tool Cache Related Errors

One category of errors was caused by failures in the gnomAD tool call.

Because the pipeline caches tool execution results, an error from one gnomAD request could be stored in the cache. When another variant later requested the same cached result, it would receive the previous error instead of retrying the tool call.

This suggests that the current tool caching mechanism needs improvement. In particular, failed tool calls should likely not be cached as valid results, and the system should distinguish between successful tool outputs and errors.

### 5.2 Judge Agent Retry Limit Errors

Another category of errors occurred when the judge agent exceeded its maximum retry limit.

This indicates that the judge agent may need improvement, potentially through:

- better prompt engineering,
- clearer output requirements,
- improved error recovery strategies.

### 5.3 Low Performance Is Unlikely to Be Caused Primarily by Tools

The poor performance on several criteria is unlikely to be explained solely by tool failures.

For example:

- BA1 and PM2_SUPPORTING both rely on gnomAD information.
- BA1 achieves excellent performance.
- PM2_SUPPORTING performs poorly.

If tool reliability were the primary issue, we would expect both criteria using the same tool to show similar performance. Instead, the difference suggests that the main problem is more likely related to:

- criterion-specific reasoning,
- instruction completeness,
- decision logic,
- or how the model interprets and applies evidence.

Therefore, further investigation should focus on the reasoning and decision-making process for individual criteria rather than only improving external tool reliability.

Two factors may lead to low performance on certain criterions. 1. The instructions that we've given to the llms are incorrect 2. llms are not following the instructions that we've give him. To investigatre whether the low performance is due to the first reason or the second one, I decide to first take a look at criterion BP7, which only has an accuracy of 0.03. 

Let's first try to find all predictions that contains BP7.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from pipeline import _safe_variant_name
from evaluation.question1 import parse_criteria_list 

gold_df = pd.read_csv("full_evaluation_dataset.csv")
gold_df = gold_df.rename(columns={"hgvs_cdna": "variant"})
gold_df["variant_key"] = gold_df["variant"].apply(_safe_variant_name)
gold_df["gold_applied"] = gold_df["gt_criteria_applied"].apply(parse_criteria_list)
gold_df = gold_df.set_index("variant_key")

folder = Path("outputs/batch_20260708_132813")  

incorrect_predictions = []

for json_file in sorted(folder.glob("*.json")):
    if json_file.name == "batch_summary.json":
        continue

    with json_file.open("r", encoding="utf-8") as f:
        payload = json.load(f)

    results = payload.get("results", {})

    variant =_safe_variant_name(payload["variant"])

    has_bp7 = any(
        result.get("criterion") == "BP7"
        for result in results
    )

    if has_bp7:
        bp7_prediction = next(
            (result for result in results if result["criterion"]=="BP7"),
            None
        )

        if bp7_prediction:
            gold_answer = "BP7" in gold_df.loc[variant, "gold_applied"]
            pred_answer = bp7_prediction.get("applies")

            if gold_answer != pred_answer:
                unmatched_prediction = {
                    "variant": variant,
                    "criterion": "BP7",
                    "evidence": bp7_prediction.get("evidence", "unknown"),
                    "reasoning": bp7_prediction.get("reasoning", "unknown"),
                    "tool_used": bp7_prediction.get("tool_used", "unknown"),
                    "prediction": pred_answer,
                    "gold_answer": gold_answer
                }

                incorrect_predictions.append(unmatched_prediction)


incorrect_df = pd.DataFrame(incorrect_predictions)
print(incorrect_df.to_string(index=False))


                    variant criterion                                                                                                                                                            evidence                                                                                                                                                                                                                                                                                                                                                                                         reasoning tool_used  prediction  gold_answer
   NM_001114753_3_c_1023C_T       BP7                                                 SpliceAI delta scores: DS_AG=0.07, DS_AL=0.01, DS_DG=0.0, DS_DL=0.0. All scores are <= 0.1 (maximum score is 0.07). According to the classification rules for BP7, the criterion applies if the variant is synonymous or intronic and all four SpliceAI delta scores are <= 0.1. The variant NM_001

## Based on the current evaluation, several observations can be made regarding the model’s behavior:

**1. The model generally follows the provided instructions**

The model appears to follow the instructions relatively well in most cases. In particular, it usually considers the required evidence sources and applies the provided decision criteria according to the given specifications.

However, one minor issue is that the reasoning section does not always explicitly mention whether the model checked important variant properties, such as whether the variant is intronic or synonymous. Since some VCEP specifications depend heavily on these variant classifications, the absence of this verification step in the reasoning may reduce transparency and make it harder to evaluate whether the model considered all necessary conditions.

**2. The reasoning is generally consistent with the evidence**

The reasoning generated by the model does not appear to contradict the evidence provided. For example, we did not observe cases where the evidence stated that SpliceAI delta scores were below 0.1, while the reasoning incorrectly claimed that the scores were greater than 1.

This suggests that the model is generally able to interpret and summarize the provided evidence correctly, rather than generating reasoning that conflicts with the underlying data.

**3. The instructions closely match the original VCEP specifications, but some exceptions may be missing**

The instructions provided to the model closely follow the original VCEP specifications. However, one important difference is that the original specification contains several explicit exceptions that are not currently included in our instructions.

For example, the original specification states:

EXCEPTION: Do NOT automatically apply BP7 if the variant is:

* a synonymous variant at the last nucleotide of an exon, or
* a deep intronic variant in the ACVRL1 intron 9 CT-rich hotspot, especially if it creates a new AG cryptic splice site, or
* from a patient/family highly suspicious for HHT with no other causative variant found.

In these cases, set applies=false or status=manual_review_needed and explain that SpliceAI alone is insufficient.

Our current instructions do not explicitly include these exception cases. As a result, the model may incorrectly apply BP7 in situations where additional manual review is required.

For example, I examine the first failed variant with chatgpt (this is very preliminary, so take it but don't trust it):

“Your CSV says this is p.Gly341Gly, so it is synonymous. Also, c.1023 is very likely an exon-terminal coding position for ENG, meaning this is the kind of synonymous variant where the HHT paper says not to rely only on computational prediction. Note that this is not strictly verified though.”

This suggests that the model was able to recognize the potential issue, but the instructions were not precise enough to force the correct decision behavior. Therefore, incomplete instructions may be one possible reason contributing to model failures.

However, this hypothesis requires further investigation with additional examples before drawing a firm conclusion.

**4. Further investigation is needed**

Overall, the current analysis suggests that the model’s failures are unlikely to be caused by a general inability to follow instructions or interpret evidence. Instead, they may result from missing edge cases in the provided specifications, particularly cases where the VCEP guidelines contain specific exceptions.

A more detailed examination of failed cases is still needed to determine the exact causes of errors. Due to time constraints, I was not able to perform a deeper analysis today.